In [ ]:
# Estrategia Lay 0x2

import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv("../../data_total/dados_betfair.csv", sep=";")

In [4]:
# Filtar colunas para análise
datatest = data[['League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Min_Goals_H', 'Odd_H_Back', 'Odd_A_Back', 'Odd_CS_0x2_Lay']].copy()

#datatest.to_csv("TEBF002_Lay_0x1.csv", sep=";", index=False)

# Verificar se o placar FT foi 0x2
datatest['WCS'] = datatest.apply(lambda row: 0 if row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 2 else 1, axis=1)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_CS_0x2_Lay'] - 1) if row['WCS'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

# Adicionar coluna com o minuto do primeiro gol
datatest['Min_Goal_0x0'] = np.where(
    (datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 0),
    datatest['Min_Goals_H'].apply(lambda x: x[1:3] if len(x) > 0 else 0),
    0
)

# Alterar [ para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].str.replace(']', '0')

# Alterar NaN para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].fillna(0)

# Transformar a coluna em inteiro
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].astype(int)


datatest.head(15)

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Min_Goals_H,Odd_H_Back,Odd_A_Back,Odd_CS_0x2_Lay,WCS,Profit,Min_Goal_0x0
0,ENGLAND 1,Fulham,Tottenham,1,0,3,0,"[42, 49, 61]",3.50,2.10,16.5,1,0.94,0
1,SPAIN 1,Osasuna,Real Madrid,1,2,2,4,"[7, 90]",6.60,1.58,9.0,1,0.94,0
2,SPAIN 1,Mallorca,Granada CF,0,0,1,0,[85],1.88,5.30,38.0,1,0.94,85
3,SPAIN 1,Getafe,Girona,1,0,1,0,[33],3.35,2.34,15.0,1,0.94,0
4,SPAIN 1,Ath Bilbao,Alaves,2,0,2,0,"[32, 37]",1.59,7.60,60.0,1,0.94,0
5,ENGLAND 1,Luton,Nottingham,0,1,1,1,[89],2.88,2.56,18.5,1,0.94,0
6,ENGLAND 1,Burnley,Brentford,1,0,2,1,"[10, 62]",3.35,2.26,15.0,1,0.94,0
7,ITALY 1,Frosinone,Lazio,1,1,2,3,"[13, 70]",3.55,2.24,14.5,1,0.94,0
8,ITALY 1,Salernitana,Lecce,0,1,0,1,[],3.10,2.60,16.0,1,0.94,0
9,ITALY 1,Monza,Cagliari,1,0,1,0,[41],2.12,3.85,28.0,1,0.94,0


In [5]:
# Função para criar faixas de odds
def criar_faixa_h_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_a_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
        
def criar_faixa_cs_lay(odd):
    if odd < 9.0:
        return '8.0-8.9'
    elif odd < 10.0:
        return '9.0-9.9'
    elif odd < 11.0:
        return '10.0-10.9'
    elif odd < 12.0:
        return '11.0-11.9'
    elif odd < 13.0:
        return '12.0-12.9'
    elif odd < 14.0:
        return '13.0-13.9'
    elif odd < 15.0:
        return '14.0-14.9'
    elif odd < 16.0:
        return '15.0-15.9'
    elif odd < 18.0:
        return '16.0-17.9'
    elif odd < 20.0:
        return '18.0-19.9'
    else:
        return '20.0+'
    
# Aplicar as funções às colunas correspondentes
datatest['Faixa_Odd_H_Back'] = datatest['Odd_H_Back'].apply(criar_faixa_h_back)
datatest['Faixa_Odd_A_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_a_back)
datatest['Faixa_Odd_CS_0x2_Lay'] = datatest['Odd_CS_0x2_Lay'].apply(criar_faixa_cs_lay)

# Agrupars por faixas e calcular estatísticas
print("\n📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):")
print("-" * 80)

agrupando_faixas = datatest.groupby(['Faixa_Odd_H_Back', 'Faixa_Odd_A_Back', 'Faixa_Odd_CS_0x2_Lay']).agg(
    Total_Jogos=('WCS', 'count'),
    Jogos_0x2=('WCS', 'sum'),
    Percentual_Acerto=('WCS', lambda x: (x.sum() / len(x) * 100)),
    Lucro_Total=('Profit', 'sum')
).reset_index()

# Arredondar valores
agrupando_faixas['Percentual_Acerto'] = agrupando_faixas['Percentual_Acerto'].round(2)

# Agrupar por lucro Total
agrupando_faixas = agrupando_faixas.sort_values(by='Lucro_Total', ascending=False)

agrupando_faixas.head(10)



📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):
--------------------------------------------------------------------------------


,Faixa_Odd_H_Back,Faixa_Odd_A_Back,Faixa_Odd_CS_0x2_Lay,Total_Jogos,Jogos_0x2,Percentual_Acerto,Lucro_Total
3,1.50-1.79,4.00-4.99,20.0+,44,44,100.00,41.36
62,3.50-3.99,2.10-2.49,15.0-15.9,18,18,100.00,16.92
75,4.00-4.99,1.80-2.09,10.0-10.9,29,28,96.55,16.82
16,2.10-2.49,2.50-2.99,20.0+,14,14,100.00,13.16
47,3.00-3.49,2.50-2.99,16.0-17.9,82,78,95.12,11.82
41,3.00-3.49,2.10-2.49,15.0-15.9,44,42,95.45,10.98
79,4.00-4.99,1.80-2.09,14.0-14.9,11,11,100.00,10.34
103,5.00+,1.80-2.09,9.0-9.9,30,28,93.33,9.92
44,3.00-3.49,2.10-2.49,20.0+,9,9,100.00,8.46
53,3.50-3.99,1.80-2.09,15.0-15.9,9,9,100.00,8.46


In [6]:
# Selecionar apenas as 8 linhas com maior lucro total
top_10_lucro = agrupando_faixas.head(10)

# Total de jogos
total_jogos = top_10_lucro['Total_Jogos'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = top_10_lucro['Jogos_0x2'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🎯 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = top_10_lucro['Lucro_Total'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")

📈 Total de Jogos: 290
✅ Total de Acertos: 281
🎯 Percentual de Acerto: 96.90%
💰 Total de Lucro: 148.24
